# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [21]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [22]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [23]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [24]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [25]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [26]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [27]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [28]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [29]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [30]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times in the list of projects.'

In [31]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project titled "MediMind 17" focuses on security with a description of "A medical imaging solution improving early diagnosis through vision transformers."'

In [32]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive view of the fintech projects, often highlighting their technical strength and real-world impact. For example, one project was described as a "clever solution with measurable environmental benefit," and another received a comment noting a "promising idea with robust experimental validation." Additionally, some projects were praised for their technical maturity, quality, and potential for impact, such as "solid work with impressive real-world impact" and "strong quantitative results." Overall, the judges recognized the innovative and effective nature of the fintech projects, though some noted minor issues like needing more benchmarking or integration improvements.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [33]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [34]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [35]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly specified for all entries, but from the sample, the domains mentioned include Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. Since there is limited data and no clear indication of the frequency of each domain across all projects, I cannot definitively determine the most common project domain.'

In [36]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases about security mentioned.'

In [37]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges' comments on the fintech projects indicate that they found the projects to be technically ambitious and well-executed."

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

BM25 (Best Matching 25) is a retriever based on the bag-of-words model, which treats a document as an unordered collection of words. It uses natural language processing and information retrieval techniques, disregards word order, and takes into account word frequency as a signal of importance. Because of this, BM25 is very effective for term-matching queries where the presence of specific keywords matters. In contrast, embeddings models focus on capturing semantic meaning and generalizing across related concepts, which can sometimes overlook documents that contain the exact keywords you are looking for.

 Example query: “Which projects use Mamba-style attention?"
 
BM25 will look for documents containing the word "Mamba-style attention" and rank them based on how frequently the term appears and the overall document length. This ensures that documents explicitly mentioning the keyword are returned. An embeddings model might return documents about modeling or general attention use that do not explicitly contain the word "Mamba-style,” which could be less precise if the goal is to find exact keyword matches. Therefore, BM25 is better than embeddings for queries that rely on exact term matching, rare keywords, or domain-specific acronyms, because it directly searches for the presence of those terms rather than their semantic meaning.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [38]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [39]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [40]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Security," as it is explicitly mentioned in one of the project entries. However, since only a small sample is shown, and no comprehensive frequency analysis is provided, I cannot definitively determine the most common project domain from this snippet alone. \n\nIf you have access to the full dataset, you could perform a count of each domain to identify the most prevalent one.'

In [41]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicitly mentioned use cases related to security. The examples focus on federated learning toolkits aimed at improving privacy in healthcare applications, but do not specify use cases directly addressing security issues.'

In [42]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. Specifically, for the project "Pathfinder 27" in the Finance / FinTech domain, the judge praised it for "Excellent code quality and use of open-source libraries" with a high judge score of 9.8. Additionally, for "PlanPilot 35," which is also related to finance/fintech, the judge called it "A clever solution with measurable environmental benefit," indicating recognition of its innovative approach.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [43]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [44]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [45]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Writing & Content," which appears multiple times among the listed projects.'

In [46]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "InsightAI 36" falls within the Security domain. It involves a synthetic data generator for low-resource domain adaptation tasks, which can be important for privacy and security in data handling and model training.'

In [47]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had a generally positive view of the fintech-related projects. For example:\n\n- The project "SecureNest" received a judge comment stating it was "conceptually strong but results need more benchmarking," and it scored a high 9.0 out of 10.\n- "Pathfinder 27," focused on finance/fintech, was praised for "excellent code quality and use of open-source libraries," and received an impressive judge score of 9.8.\n- "CreateFlow," another fintech project related to privacy in healthcare, was noted as "a clever solution with measurable environmental benefit" with a score of 8.4.\n- "DocuCheck," which also sits within the fintech domain, was described as "conceptually strong but results need more benchmarking," with a high judge score of 9.6.\n\nOverall, judges recognized the strength, innovation, and potential impact of these fintech projects, although some noted areas for further validation or benchmarking.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

In the real world, user queries can vary widely in wording and level of specificity. Generating multiple reformulations of a user query can improve recall by creating a larger and broader pool of different ways to retrieve relevant information. This approach helps catch synonyms, alternative phrasings, and differences in terminology that might otherwise cause important documents to be missed. By expanding the ways a query is interpreted, the system increases the chances of finding all relevant documents. The trade-off is that this can increase computational cost and latency, since multiple queries need to be processed instead of just one.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [48]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [49]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [50]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [51]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [52]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [53]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the project domains mentioned include Security, Productivity Assistants, Creative / Design / Media, and Healthcare / MedTech. Since the sample contains only a few entries, I cannot definitively determine the most common project domain across all data. However, from this sample, each domain appears only once. Therefore, I do not have enough information to identify the most common project domain.'

In [54]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific mentions of use cases related to security. The projects mainly focus on federated learning to improve privacy, especially in healthcare applications, but security as a distinct use case is not explicitly discussed.'

In [55]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about some of the projects. For example, they described "Pathfinder 25" as a "promising idea with robust experimental validation," and "CreateFlow" as "solid work with impressive real-world impact." Overall, the judges seemed to appreciate the innovation and potential impact of the projects, though specific comments about fintech projects are limited.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [56]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [57]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [58]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be Healthcare / MedTech, as it is mentioned multiple times among the projects listed.'

In [59]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "MediMind" falls under the Security domain and involves a medical imaging solution aimed at improving early diagnosis through vision transformers.'

In [60]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive comments about the fintech projects. For example, they described the project "PulseAI" as "Technically ambitious and well-executed," and "PlanPilot 35" as "A clever solution with measurable environmental benefit." Overall, the judges recognized the innovative nature and potential of the fintech projects, highlighting their technical quality and practical impact.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [61]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [62]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [63]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [64]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [65]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [66]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice among the listed projects.'

In [67]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security in the provided context. Specifically, projects such as "MediMind" (BioForge) and "InsightAI" (Project Aurora) are categorized under the "Security" domain.'

In [68]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive comments about the fintech projects. For example, they described the projects as "comprehensive and technically mature," "technically ambitious and well-executed," and having "solid supporting data" with "impressive real-world impact." Overall, the judges recognized the projects for their technical quality, ambition, and potential for real-world application.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

Semantic chunking is the process of splitting or combining sentences based on their semantic meaning, which is the underlying meaning or concept conveyed by the text rather than the exact words. When sentences are short and highly repetitive, such as in FAQs, semantic chunking may produce many small chunks with minimal semantic differences, leading to duplicate or redundant chunks that increase processing cost. Conversely, it might combine several sentences into a larger chunk, which could be unfocused and lose the granularity needed for precise retrieval.

To adjust the algorithm in such cases, you could add steps to remove duplicate chunks, perform additional pre-processing to improve the data’s suitability for chunking, use a hybrid method (i.e. use the structure of the document as well), or set explicit limits on chunk size. However, it may be the case that semantic chunking is not the ideal approach for extremely short and repetitive text, and an alternative strategy might be more effective.



# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

# ✅🎉 **COMPLETED ACTIVITY IS IN [`breakout2_activity.ipynb`](https://github.com/bzgold/AIE8_bg/blob/week5-activity1/09_Advanced_Retrieval/Breakout2%20Activity.ipynb)** 🎉✅

---

> ⚠️ **Note:**  
> The code below is another run for testing and iteration.  
> The **most complete and finalized output** can be viewed here:  
> 👉 [`breakout2_activity.ipynb`](https://github.com/bzgold/AIE8_bg/blob/week5-activity1/09_Advanced_Retrieval/Breakout2%20Activity.ipynb)

LINK: https://github.com/bzgold/AIE8_bg/blob/week5-activity1/09_Advanced_Retrieval/Breakout2%20Activity.ipynb

---


In [1]:
### YOUR CODE HERE
# Section 1: Imports
import os
from getpass import getpass
import pandas as pd
from operator import itemgetter
import time
import random
import pypdf

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import Qdrant
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Retrievers
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import (
    MultiQueryRetriever,
    ContextualCompressionRetriever,
    EnsembleRetriever,
    ParentDocumentRetriever
)
from langchain.retrievers.document_compressors import CohereRerank
from langchain.storage import InMemoryStore

# Ragas (0.2.10) - Different API than 0.3.x!
from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy, ContextEntityRecall
from ragas.testset import TestsetGenerator
from datasets import Dataset  # For Ragas 0.2.10 evaluation format

# LangSmith
from langsmith import Client

print("✅ All imports successful")



✅ All imports successful


In [2]:
# API Keys Setup
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["COHERE_API_KEY"] = getpass("Enter Cohere API Key: ")
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangChain/LangSmith API Key: ")

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Advanced_Retrieval_Evaluation"

langsmith_client = Client()
print("✅ API keys configured and LangSmith tracing enabled")


✅ API keys configured and LangSmith tracing enabled


In [3]:
# Section 2: Load PDF
pdf_path = "data/howpeopleuseai.pdf"
print(f"📄 Loading PDF: {pdf_path}")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from PDF")
print(f"   First page preview: {documents[0].page_content[:200]}...")

# Split for retrieval
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"✅ Split into {len(chunks)} chunks for retrieval")


📄 Loading PDF: data/howpeopleuseai.pdf
✅ Loaded 64 pages from PDF
   First page preview: NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/...
✅ Split into 159 chunks for retrieval


In [4]:
# Section 3: Golden Dataset Generation (Ragas SDG 0.2.10) - 10 Questions
# Using the EXACT working approach from Assignment 07

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(documents, testset_size=10)

# Convert to pandas for easy access
golden_df = golden_dataset.to_pandas()

print(f"✅ Generated {len(golden_df)} synthetic questions!")
print("\\n📋 Sample questions:")
for i in range(min(3, len(golden_df))):
    print(f"   {i+1}. {golden_df.iloc[i]['user_input']}")

# Display full dataset
golden_df


Applying HeadlinesExtractor:   0%|          | 0/22 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/41 [00:00<?, ?it/s]

Property 'summary' already exists in node 'ff03f5'. Skipping!
Property 'summary' already exists in node '9997ff'. Skipping!
Property 'summary' already exists in node 'f92590'. Skipping!
Property 'summary' already exists in node '65640c'. Skipping!
Property 'summary' already exists in node 'a127b2'. Skipping!
Property 'summary' already exists in node '6c9cd1'. Skipping!
Property 'summary' already exists in node '16a8fb'. Skipping!
Property 'summary' already exists in node '788f44'. Skipping!
Property 'summary' already exists in node '30abce'. Skipping!
Property 'summary' already exists in node 'f381c9'. Skipping!
Property 'summary' already exists in node 'b4d0c4'. Skipping!
Property 'summary' already exists in node '4cb0a4'. Skipping!
Property 'summary' already exists in node '2fe0f1'. Skipping!
Property 'summary' already exists in node 'c45e9f'. Skipping!
Property 'summary' already exists in node '1fc0df'. Skipping!
Property 'summary' already exists in node '381797'. Skipping!
Property

...

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Generated 12 synthetic questions!
\n📋 Sample questions:
   1. What significant milestones in user engagement with ChatGPT were reached by July 2025?
   2. What insights does Acemoglu provide regarding the impact of AI on economic growth?
   3. How does the usage of ChatGPT vary among engineering and science occupations compared to other professional categories?


,user_input,reference_contexts,reference,synthesizer_name
0,What significant milestones in user engagement...,[Introduction ChatGPT launched in November 202...,"By July 2025, ChatGPT had reached a remarkable...",single_hop_specifc_query_synthesizer
1,What insights does Acemoglu provide regarding ...,[Introduction ChatGPT launched in November 202...,Acemoglu's work discusses the intensified inte...,single_hop_specifc_query_synthesizer
2,How does the usage of ChatGPT vary among engin...,[Variation by Occupation Figure 23 presents va...,"In the context of ChatGPT usage, engineering a...",single_hop_specifc_query_synthesizer
3,How does ChatGPT usage vary by occupation?,[Variation by Occupation Figure 23 presents va...,ChatGPT usage varies significantly by occupati...,single_hop_specifc_query_synthesizer
4,What are the user adoption statistics for Chat...,[<1-hop>\n\nConclusion This paper studies the ...,"As of July 2025, ChatGPT had been used weekly ...",multi_hop_abstract_query_synthesizer
5,What are the implications of the rapid growth ...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth of ChatGPT, which saw over 70...",multi_hop_abstract_query_synthesizer
6,What evidence supports the claim that the rapi...,[<1-hop>\n\nConclusion This paper studies the ...,The evidence supporting the claim that the rap...,multi_hop_abstract_query_synthesizer
7,How does the rapid adoption of ChatGPT impact ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The rapid adoption of ChatGPT, which saw 700 m...",multi_hop_abstract_query_synthesizer
8,What were the key statistics regarding ChatGPT...,[<1-hop>\n\nConclusion This paper studies the ...,"By July 2025, ChatGPT had achieved remarkable ...",multi_hop_specific_query_synthesizer
9,How many users were using ChatGPT by July 2025...,[<1-hop>\n\nConclusion This paper studies the ...,"By July 2025, ChatGPT had been used by 700 mil...",multi_hop_specific_query_synthesizer


In [5]:
# Section 4: Setup All 6 Retrievers
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Naive
vectorstore = Qdrant.from_documents(
    chunks,
    embeddings,
    location=":memory:",
    collection_name="howpeopleuseai"
)
naive = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. BM25
bm25 = BM25Retriever.from_documents(chunks); bm25.k = 5

# 3. Multi-Query
multi_query = MultiQueryRetriever.from_llm(retriever=naive, llm=llm)

# 4. Parent Document
store = InMemoryStore()
child_chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(documents)
parent_doc = ParentDocumentRetriever(
    vectorstore=Qdrant.from_documents(
        child_chunks,
        embeddings,
        location=":memory:",
        collection_name="parent_chunks"
    ),
    docstore=store,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
)
parent_doc.add_documents(documents)

# 5. Contextual Compression (Cohere Rerank)
compression = ContextualCompressionRetriever(
    base_compressor=CohereRerank(model="rerank-english-v3.0", top_n=5),
    base_retriever=naive
)

# 6. Ensemble
ensemble = EnsembleRetriever(retrievers=[bm25, naive, multi_query], weights=[0.3, 0.4, 0.3])

retrievers = {"Naive": naive, "BM25": bm25, "Multi-Query": multi_query, 
              "Parent Document": parent_doc, "Contextual Compression": compression, "Ensemble": ensemble}

print(f"✅ All {len(retrievers)} retrievers ready")


✅ All 6 retrievers ready


/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_69747/2653236264.py:38: LangChainDeprecationWarning: The class `CohereRerank` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-cohere package and should be used instead. To use it run `pip install -U :class:`~langchain-cohere` and import as `from :class:`~langchain_cohere import CohereRerank``.
  base_compressor=CohereRerank(model="rerank-english-v3.0", top_n=5),


In [6]:
# Section 5: Build RAG Chains with LangSmith Tracking
RAG_TEMPLATE = """You are a helpful AI assistant. Use the context to answer the question.
If you don't know, say so. Don't make up information.

Question: {question}
Context: {context}
Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def build_chain(retriever):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=lambda x: "\\n\\n".join(d.page_content for d in x["context"]))
        | {"response": rag_prompt | rag_llm, "context": itemgetter("context")}
    )

rag_chains = {name: build_chain(r) for name, r in retrievers.items()}
print(f"✅ {len(rag_chains)} RAG chains ready with LangSmith tracking")


✅ 6 RAG chains ready with LangSmith tracking


In [ ]:
# Section 6: Evaluate All Retrievers with Ragas Metrics (0.2.10 API)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
metrics = [ContextPrecision(), ContextRecall(), AnswerRelevancy(), Faithfulness(), ContextEntityRecall()]

print(f"📊 Evaluating {len(retrievers)} retrievers with {len(metrics)} Ragas metrics...")
evaluation_results = {}

for name, chain in rag_chains.items():
    print(f"\\n{'='*50}\\nEvaluating: {name}\\n{'='*50}")
    start = time.time()
    
    # Build evaluation data (Ragas 0.2.10 format)
    # Required columns: question, answer, contexts, ground_truths, reference
    eval_data = {"question": [], "answer": [], "contexts": [], "ground_truths": [], "reference": []}
    
    # Use enumerate to get proper sequential count
    for idx in range(len(golden_df)):
        row = golden_df.iloc[idx]
        print(f"  Question {idx+1}/{len(golden_df)}...", end="\\r")
        try:
            question = row['user_input']
            reference = row.get('reference', '')
            
            result = chain.invoke({"question": question})
            
            eval_data["question"].append(question)
            eval_data["answer"].append(result["response"].content)
            eval_data["contexts"].append([result["context"]])
            eval_data["ground_truths"].append([reference] if reference else [question])
            eval_data["reference"].append(reference if reference else question)
            
            time.sleep(0.3)
        except Exception as e:
            print(f"\\n  ✗ Error on question {idx+1}: {e}")
            continue  # Skip failed questions
    
    print(f"\\n  Running Ragas evaluation...")
    from datasets import Dataset
    eval_dataset = Dataset.from_dict(eval_data)
    ragas_results = evaluate(eval_dataset, metrics=metrics, llm=evaluator_llm)
    elapsed = time.time() - start
    
    evaluation_results[name] = {"ragas": ragas_results, "latency": elapsed, "count": len(eval_data["question"])}
    print(f"  ✅ Complete ({elapsed:.1f}s) - Evaluated {len(eval_data['question'])}/{len(golden_df)} questions")

print("\\n" + "="*70)
print("📊 EVALUATION COMPLETE - QUICK SUMMARY")
print("="*70)

# Quick summary table
summary_data = []
for name, res in evaluation_results.items():
    ragas_result = res["ragas"]
    row = {
        "Retriever": name,
        "Questions": res["count"],
        "Latency (s)": f"{res['latency']:.1f}"
    }
    # Add first few metrics for quick view
    for metric in metrics[:3]:  # Show first 3 metrics
        if hasattr(ragas_result, metric.name):
            row[metric.name] = f"{getattr(ragas_result, metric.name):.3f}"
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("\\n✅ Full detailed results in next cell...")


📊 Evaluating 6 retrievers with 5 Ragas metrics...
\n==================================================\nEvaluating: Naive\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (98.3s) - Evaluated 12/12 questions
\n==================================================\nEvaluating: BM25\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (94.8s) - Evaluated 12/12 questions
\n==================================================\nEvaluating: Multi-Query\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


...

  ✅ Complete (85.0s) - Evaluated 12/12 questions
\n==================================================\nEvaluating: Contextual Compression\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r\n  ✗ Error on question 11: status_code: 429, body: data=None id='5778b7bf-cc2b-4bfb-b938-8ac6c52d78c6' message="You are using a Trial key, which is limited to 10 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at support@cohere.com with any questions"
  Question 12/12...\r\n  ✗ Error on question 12: status_code: 429, body: data=None id='19270820-47a3-42df-bfe1-1701921f61b3' message="You are using a Trial key

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Complete (100.6s) - Evaluated 10/12 questions
\n==================================================\nEvaluating: Ensemble\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (257.1s) - Evaluated 12/12 questions
\n======================================================================
📊 EVALUATION COMPLETE - QUICK SUMMARY
             Retriever  Questions Latency (s)
                 Naive         12        98.3
                  BM25         12        94.8
           Multi-Query         12       119.3
       Parent Document         12        85.0
Contextual Compression         10       100.6
              Ensemble         12       257.1
\n✅ Full detailed results in next cell...


In [17]:
# Quick Summary: Display all Ragas results from Section 6
# NOTE: This cell just DISPLAYS results - it doesn't re-run evaluation!
# Run this cell anytime after Section 6 completes to see the results instantly.

print("="*70)
print("📊 RAGAS EVALUATION RESULTS - ALL RETRIEVERS")
print("="*70)

# Define metric names we want to display
metric_names = ['context_precision', 'context_recall', 'answer_relevancy', 'faithfulness', 'context_entity_recall']

for retriever_name, result in evaluation_results.items():
    print(f"\n🔹 {retriever_name}")
    print(f"   Latency: {result['latency']:.1f}s | Questions: {result['count']}")
    print("   Metrics:")
    
    ragas_result = result["ragas"]
    for metric_name in metric_names:
        if hasattr(ragas_result, metric_name):
            score = getattr(ragas_result, metric_name)
            print(f"      • {metric_name}: {score:.4f}")

print("\n" + "="*70)


📊 RAGAS EVALUATION RESULTS - ALL RETRIEVERS

🔹 Naive
   Latency: 98.3s | Questions: 12
   Metrics:

🔹 BM25
   Latency: 94.8s | Questions: 12
   Metrics:

🔹 Multi-Query
   Latency: 119.3s | Questions: 12
   Metrics:

🔹 Parent Document
   Latency: 85.0s | Questions: 12
   Metrics:

🔹 Contextual Compression
   Latency: 100.6s | Questions: 10
   Metrics:

🔹 Ensemble
   Latency: 257.1s | Questions: 12
   Metrics:



In [14]:
# Section 7: Results Compilation and Analysis
results_data = []
for name, res in evaluation_results.items():
    # Ragas 0.2.10: EvaluationResult has metric scores as attributes
    ragas_result = res["ragas"]
    row = {
        "Retriever": name, 
        "Latency (s)": res["latency"], 
        "Questions": res["count"]
    }
    
    # Extract metric scores from EvaluationResult
    for metric in metrics:
        metric_name = metric.name
        if hasattr(ragas_result, metric_name):
            row[metric_name] = getattr(ragas_result, metric_name)
    
    results_data.append(row)

results_df = pd.DataFrame(results_data).sort_values("Latency (s)")
print("\\n📊 FINAL RESULTS:")
print(results_df.to_string(index=False))

results_df.to_csv("retriever_evaluation_results.csv", index=False)
print("\\n✅ Saved to: retriever_evaluation_results.csv")

# Analysis
best = {col: results_df.loc[results_df[col].idxmax(), "Retriever"] 
        for col in results_df.columns if col not in ["Retriever", "Latency (s)", "Questions"]}

print("\\n🏆 Best by Metric:")
for metric, retriever in best.items():
    print(f"   {metric}: {retriever}")

print("\\n📝 RECOMMENDATION:")
print("""
Based on cost, latency, and performance:
- For best performance: Check scores above
- For best speed: Check latency column  
- For balanced approach: Consider all factors
""")

# LangSmith URLs
project_name = "Advanced_Retrieval_Evaluation"
print("\\n🔗 LangSmith Links:")
print(f"   📊 Project Dashboard: https://smith.langchain.com/o/default/projects/p/{project_name.replace('_', '-').lower()}")
print(f"   🔍 View All Traces: https://smith.langchain.com/")
print(f"   📁 Project: {project_name}")
print("\\n   💡 Tip: Use the LangSmith dashboard to:")
print("      - View detailed traces for each retriever")
print("      - Compare latency and token usage")
print("      - Debug retrieval quality issues")
print("      - Analyze cost breakdowns")

results_df


\n📊 FINAL RESULTS:
             Retriever  Latency (s)  Questions
       Parent Document    85.037315         12
                  BM25    94.754703         12
                 Naive    98.255482         12
Contextual Compression   100.646581         10
           Multi-Query   119.260719         12
              Ensemble   257.126659         12
\n✅ Saved to: retriever_evaluation_results.csv
\n🏆 Best by Metric:
\n📝 RECOMMENDATION:

Based on cost, latency, and performance:
- For best performance: Check scores above
- For best speed: Check latency column  
- For balanced approach: Consider all factors

\n🔗 LangSmith Links:
   📊 Project Dashboard: https://smith.langchain.com/o/default/projects/p/advanced-retrieval-evaluation
   🔍 View All Traces: https://smith.langchain.com/
   📁 Project: Advanced_Retrieval_Evaluation
\n   💡 Tip: Use the LangSmith dashboard to:
      - View detailed traces for each retriever
      - Compare latency and token usage
      - Debug retrieval quality issues
  

,Retriever,Latency (s),Questions
3,Parent Document,85.037315,12
1,BM25,94.754703,12
0,Naive,98.255482,12
4,Contextual Compression,100.646581,10
2,Multi-Query,119.260719,12
5,Ensemble,257.126659,12
